In [1]:
# ---------- Setup (deine Pipeline vom Credit-Scoring, unverändert übernommen) ----------
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
np.set_printoptions(suppress=True)

# Annahmen wie im Notebook
test_anteil = 0.3
seed = 42

# Daten laden, Ziel 0/1
daten = fetch_openml("credit-g", version=1, as_frame=True)
df = daten.frame
df['ausfall'] = df['class'].map({'good': 0, 'bad': 1}).astype(int)

# Split ohne Datenleck, stratifiziert
y = df['ausfall']
X = df.drop(columns=['class', 'ausfall'])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_anteil, random_state=seed, stratify=y)

# One-Hot-Codierung
X_train_k = pd.get_dummies(X_train, dtype=int)
X_test_k  = pd.get_dummies(X_test,  dtype=int)

# Standardisierung nur der Zahlenspalten, fit nur auf Train
scaler = StandardScaler()
X_train_s = X_train_k.copy()
X_test_s  = X_test_k.copy()
num_cols = X_train.select_dtypes(include='number').columns
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols]  = scaler.transform(X_test[num_cols])

# NumPy-Arrays für die Netz-Rechnung
Xtr = X_train_s.values;  ytr = y_train.values
Xte = X_test_s.values;   yte = y_test.values
print(Xtr.shape, ytr.shape, Xte.shape, yte.shape)   # Kontrollwert: (700, 61) (700,) (300, 61) (300,)
print(yte.mean().round(3))                           # Kontrollwert: 0.3 (30 % Ausfälle)

(700, 61) (700,) (300, 61) (300,)
0.3


In [2]:
# Zufallsgewichte — Generator mit festem Seed, damit es reproduzierbar ist
rng = np.random.default_rng(42)
# Beispiel für eine Gewichtsmatrix: rng.normal(size=(zeilen, spalten)) * 0.1
# Beispiel für einen Bias-Vektor:   rng.normal(size=(spalten,)) * 0.1

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def ReLU(z):
    return np.maximum(0, z)

W1 = rng.normal(loc=0, scale=0.1, size=(61,4))
b1 = rng.normal(loc=0, scale=0.1, size=(4, ))

Z1 = Xte @ W1 + b1
A1 = ReLU(Z1)
print(A1.shape)
print((A1 >= 0).all())

W2 = rng.normal(loc=0, scale=0.1, size=(4,1))
b2 = rng.normal(loc=0, scale=0.1, size=(1, ))

Z2 = A1 @ W2 + b2
A2 = sigmoid(Z2)
print((A2.min() >= 0) & (A2.max() <= 1))
print(A2.shape)

pred = (A2.ravel() >= 0.5).astype(int)
print(pred.shape)

Accuracy = (pred == yte).mean()
print(f'Anteil Vorhersage Klasse 1: {pred.mean()} Tatsächlicher Anteil der Klasse 1: {yte.mean()}')
print(Accuracy)


(300, 4)
True
True
(300, 1)
(300,)
Anteil Vorhersage Klasse 1: 1.0 Tatsächlicher Anteil der Klasse 1: 0.3
0.3


In [13]:
import numpy as np
rng = np.random.default_rng(42)           # eigener Zufallsgenerator, Seed 42
for faktor in [1, 0.1]:
    W1 = rng.normal(size=(61, 4)) * faktor; b1 = np.zeros(4)
    W2 = rng.normal(size=(4, 1)) * faktor;  b2 = np.zeros(1)
    Z1 = Xte @ W1 + b1;  A1 = np.maximum(Z1, 0)             # Schicht 1, ReLU
    Z2 = A1 @ W2 + b2;   A2 = 1 / (1 + np.exp(-Z2))          # Schicht 2, Sigmoid
    print(f"Faktor {faktor}: Z2 von {Z2.min():.1f} bis {Z2.max():.1f} | "
          f"A2 von {A2.min():.3f} bis {A2.max():.3f} | Anteil A2 > 0,5: {(A2 > 0.5).mean():.2f}")

Faktor 1: Z2 von -14.0 bis 10.8 | A2 von 0.000 bis 1.000 | Anteil A2 > 0,5: 0.08
Faktor 0.1: Z2 von -0.0 bis 0.2 | A2 von 0.494 bis 0.549 | Anteil A2 > 0,5: 0.88


### Zerlegung TF-Nachbau (Tag 70, 04.11.2026)
1. Layer 1 (hidden): aus Xte (300, 61) über W1 (61, 4) und b1 (4,) → Z1 (300, 4), A1 = ReLU(Z1) (300, 4)
2. Layer 2 (Output): aus A1 über W2 (4, 1) und b2 (1,) → Z2 (300, 1), A2 = sigmoid(Z2) (300, 1)
3. A2 ist die Wahrscheinlichkeit → Decision Boundary 0,5 → binäres Array (300,)
In TensorFlow: Schritt 1 und 2 = zwei Dense-Schichten; meine Gewichte mit set_weights

In [16]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

# (1) Modell: Input sagt TF die 61 Eingänge, dann deine zwei Schichten
model = Sequential([
    Input(shape=(61,)),
    Dense(4, activation='relu'), # = Z1 = X @ W1 + b1, A1 = ReLU(Z1)
    Dense(1, activation='sigmoid')]) # = Z2 = A1 @ W2 + b2, A2 = sigmoid(Z2)
model.summary() # Kontrollwert: 248 + 5 = 253 Parameter

# (2) deine Gewichte hineinsetzen — Liste [W, b], Formen müssen passen
model.layers[0].set_weights([W1, b1]) # W1 (61, 4), b1 (4,)
model.layers[1].set_weights([W2, b2]) # W2 (4, 1),  b2 (1,)

# (3) vorwärts rechnen
A2_tf = model.predict(Xte, verbose=0)
print(A2_tf.shape) # Kontrollwert: (300, 1)

# (4) Vergleich mit deinem NumPy-A2
print(np.abs(A2_tf - A2).max()) # Kontrollwert: kleiner als 0,00001

# (5) Schwelle 0,5 → Klassen, Accuracy
pred_tf = (A2_tf.ravel() >= 0.5).astype(int)
print((pred_tf == yte).mean())  # Kontrollwert: 0.3

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 4)              │           248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 253 (1012.00 B)

 Trainable params: 253 (1012.00 B)

 Non-trainable params: 0 (0.00 B)

(300, 1)
8.811530394225997e-08
0.29


# Lernnotiz Tag 69 (Di 03.11.2026) — Neuronale Netze

Erkenntnisse, Schlussfolgerungen und Erläuterungen aus dem Denkmodell-Vorblock, den Videos (Ng Course 2, Woche 1, Abschnitte „Neural networks intuition" + „Neural network model"), dem Lab „Neurons and Layers" und dem Bau mit Gerüst. Ergänzt die Unterlage `Cheat-Sheets/NN_Denkmodell.docx`. Notebook: `NN_exercise.ipynb`.

---
## 1 · Kernaussage

Ein neuronales Netz ist die logistische Regression vom 20.10. — **mehrfach nebeneinander** (= eine Schicht) und **mehrfach hintereinander** (= Tiefe). Jedes Neuron rechnet `z = X @ w + b`, dann `a = g(z)`. Neuron = (w, b, g), nichts weiter.

In Woche 1 wird nur **vorwärts** gerechnet (Forward Propagation) mit gegebenen Gewichten. Das Lernen der Gewichte (Kosten, Gradient, Training) ist Woche 2.
## 2 · Die einzige Formenregel

```
(n, e) @ (e, k)  →  (n, k)
```
Innere Zahl `e` muss gleich sein, sie verschwindet. Außen bleiben `n` (links) und `k` (rechts).

## 3 · Schritt für Schritt mit Bedeutung (Grundtabelle: 4 Kredite, 3 Merkmale)

```
SCHICHT 1   (Eingabe X = 4 Kredite × 3 Merkmale)
Schritt 1   X (4, 3) @ W1 (3, 2)   →  (4, 2)      innere 3 = Merkmale, verschwindet
                                                   übrig: 4 Kredite × 2 Neuronen (Rohwert je Neuron)
Schritt 2   (4, 2) + b1 (2,)       →  Z1 (4, 2)   b1 auf jede Zeile addiert, Form bleibt
Schritt 3   g(Z1)                  →  A1 (4, 2)   4 Kredite × 2 NEUE Merkmale (0…1)

SCHICHT 2   (Eingabe A1 = 4 Kredite × 2 neue Merkmale; X nicht mehr sichtbar)
Schritt 4   A1 (4, 2) @ W2 (2, 1)  →  (4, 1)      innere 2 = neue Merkmale, verschwindet
Schritt 5   (4, 1) + b2 (1,)       →  Z2 (4, 1)
Schritt 6   sigmoid(Z2)            →  A2 (4, 1)   4 Kredite × 1 Ausfallwahrscheinlichkeit
```

**Lesehilfe:** Zeilen = immer Kredite. Spalten = Neuronen der Schicht, die gerade gerechnet hat — in der nächsten Schicht heißen dieselben Spalten „Merkmale", weil sie dort Eingabe sind. Dieser **Rollenwechsel (Ausgabe-Neuron → Eingabe-Merkmal)** ist das ganze „hintereinander".

**Woher kommen die Formen von W und b?**
```
W = (Spalten der Eingabe dieser Schicht , Anzahl Neuronen dieser Schicht)
b = (Anzahl Neuronen dieser Schicht ,)
```
Die Zeilenzahl (Kredite) kommt in keinem W und keinem b vor — Gewichte gelten für alle Kredite gleich, sonst wäre das Modell nicht auf neue Kunden anwendbar.

**Credit-Scoring (n = 700, e = 61, k = 4):**
```
X_train_s (700, 61) @ W1 (61, 4) + b1 (4,) → Z1 (700, 4) → A1 (700, 4)
A1        (700, 4)  @ W2 (4, 1)  + b2 (1,) → Z2 (700, 1) → A2 (700, 1)
```
Deine logistische Regression im selben Schema = ein Netz mit einer Schicht und einem Neuron:
`X (700, 61) @ w (61,) + b → z (700,) → a (700,)` — eine Klammer weniger, Inhalt gleich.

## 4 · Mein Merkbild (Alp) und zwei Korrekturen

Merkbild: „Die Kredite bleiben, da sie beurteilt werden sollen. Die Merkmale werden pro Schicht über die Neuronen-Matrizen wie in einem Trichter reduziert, bis am Ende ein Neuron mit der finalen Wahrscheinlichkeit übrig bleibt."

- Korrektur 1: Es sind die **Merkmale**, die je Schicht neu zusammengesetzt werden, nicht die Gewichte. `W` wird nicht reduziert, `W` *macht* die Reduktion.
- Korrektur 2: Trichter stimmt am **Ausgang** immer (1 Neuron für ja/nein), aber in der Mitte darf das Netz **breiter** werden als die Eingabe (Ng: 3 → 25 → 15 → 1; bei dir wäre 61 → 100 → 1 erlaubt). Merksatz: *Spaltenzahl je Schicht frei wählbar, nur die letzte ist durch die Aufgabe festgelegt.*

## 5 · Nebeneinander ≠ untereinander (Stolperer aus dem Vorblock)

Zwei Neuronen-Ausgaben je `(4,)` **nebeneinander** → `(4, 2)` (`np.column_stack`), **nicht** `(8,)` (`np.concatenate`, untereinander). Nebeneinander heißt: Zeilen bleiben Kredite, es kommt eine Spalte dazu.

## 6 · Form-Stolperstein: sklearn `coef_` vs. `W`

- sklearn: `coef_` (1, 61) — eine **Zeile** je Klasse.
- Ng / TensorFlow: `W` (61, k) — eine **Spalte** je Neuron.
- Gleiche Information, gedreht: `modell.coef_.T` (61, 1) wäre das `W1` einer Schicht mit einem Neuron.

## 7 · Aktivierungsfunktionen

- **Sigmoid**: quetscht in (0, 1) → Ausgang bei ja/nein (Wahrscheinlichkeit für Schwelle und Euro-Rechnung).
- **ReLU** = `np.maximum(0, z)`: Negatives → 0, Positives bleibt → versteckte Schichten (schnell, lernt besser). Achtung `np.maximum` (elementweise), nicht `np.max` (ein Maximum).
- **Softmax**: mehrere Klassen am Ausgang (Woche 2). **linear**: Zahl am Ausgang.
- Beispiel: `z = [-0.7, 2.3]` → ReLU `[0, 2.3]`, Sigmoid ≈ `[0.33, 0.91]`.

## 8 · Was ist neu gegenüber der logistischen Regression?

| | LogistischeRegression (20.10.) | Neuronales Netz |
|---|---|---|
| Merkmale | ich baue sie (get_dummies, Scaler, 61 Spalten) | Schicht 1 baut aus 61 Spalten k neue; ich wähle nur k |
| Rechenschritt | einmal `z = X @ w + b`, `a = sigmoid(z)` | dasselbe, k-mal nebeneinander, L-mal hintereinander |
| Parameter | 62 | bei k = 4: 61·4 + 4 + 4·1 + 1 = 253 |
| Deutung | Gewicht je Spalte lesbar (Top 3 am 02.11.) | W2 wiegt namenlose Zwischengrößen — schwerer deutbar |
| Training | Kosten + Gradient Descent, selbst gebaut | gleiches Prinzip, rückwärts durch alle Schichten (Woche 2) |

Ein Netz lohnt sich, wenn Rohspalten die Antwort nicht direkt tragen, sondern erst Kombinationen (Bilder, Ton, Text). Beim German-Credit-Datensatz mit 61 handgebauten Spalten ist der Gewinn klein — die logistische Regression bleibt dort ernsthafte Alternative. Das Netz ist Grundbaustein für den LLM-Strang ab 12.11.

## 9 · Blind-Fragen (Ergebnis 2/3) — die beiden Wackler

- **Schicht vs. 5 getrennte Regressionen:** Auch die Schicht gibt 5 Zahlen je Kredit aus. Unterschied ist, *was damit passiert*: getrennt = 5 fertige Vorhersagen; in der Schicht = 5 **neue Merkmale**, die als Tabelle an die nächste Schicht gehen und dort gemeinsam gewichtet werden. Erst das Weiterreichen macht aus 5 Regressionen ein Netz.
- **ReLU auf [-0.7, 2.3]** = `[0, 2.3]` — die 0 nicht vergessen. Und: am Ausgang Sigmoid, weil Schwelle + Euro-Rechnung eine Zahl in (0, 1) brauchen.

## 10 · Ng-Notation und TensorFlow-Übersetzung (Lab „Neurons and Layers")

Notation: Schicht hochgestellt in eckigen Klammern, Neuron tiefgestellt. `a₂^[1]` = Ausgabe Neuron 2 in Schicht 1 = `A1[:, 1]`; `w₂^[1]` = `W1[:, 1]`; Eingabe x = `a^[0]`. Letzte Schicht = output layer, dazwischen = hidden layers.

```
Dense(units=1, activation='sigmoid')   eine Schicht: W (Eingänge, 1), b (1,), g = Sigmoid;
                                        Eingänge legt TF erst fest, wenn Daten durchlaufen
Sequential([schicht1, schicht2])        die Kette X → A1 → A2, wie unser forward()
modell.predict(X)                       Forward Propagation, gibt A2 zurück
schicht.get_weights()                   [W, b] — W (Eingänge, Neuronen), Spalte je Neuron
schicht.set_weights([W, b])             eigene W, b hineinlegen (im Lab, zum Nachrechnen)
```
Stolpersteine: Eingaben zweidimensional (`(1, 3)`, nicht `(3,)`) — wie bei sklearn `predict`; Rückgabe ist ein Tensor, `.numpy()` wandelt um. Stand: Ich kann sagen, was je Schritt passiert; die Syntax kommt über die TensorFlow-Befehlsreferenz (Claude, Cheat-Sheets/), nicht über Lernzeit.

## 11 · Bau mit Gerüst — Ergebnis und Schlussfolgerung

Zufallsgewichte (Seed 42, Streuung 0,1), Schicht 1: 4 Neuronen ReLU, Schicht 2: 1 Neuron Sigmoid, auf `Xte` (300, 61). Alle Formen und Bedingungen erfüllt, Denkmodell-Fehler im Code: 0.

Ergebnis: `pred.mean() = 1.0`, Accuracy **0,30**. Das Netz lehnt jeden Kredit ab — Accuracy = Anteil der tatsächlichen Ausfälle. Spiegelbild von „immer good" (0,70).

Schlussfolgerungen:
1. Mit Zufallsgewichten weiß das Netz nichts über Kredite; ob 0,30 oder 0,70 herauskommt, entscheidet allein, auf welche Seite der Schwelle die kleinen `z`-Werte kippen (ein Vorzeichen in `W2`/`b2` reicht). Münzwurf auf Modellebene.
2. **Die 0,783 des sklearn-Modells kommen zu 100 % aus dem Training, zu 0 % aus der Architektur.** Damit das Netz dorthin kommt: Kosten messen → Gradient → Gewichte anpassen, rückwärts durch beide Schichten (Woche 2).
3. Meine Erwartung „ähnliche Größenordnung" war die falsche Intuition — festhalten als Merksatz.

## 12 · Offener Punkt: Warum Anfangsgewichte klein (Faktor 0,1)?

Spalten in `Xte` haben Streuung ≈ 1; jedes Neuron summiert 61 davon. Gewichte mit Streuung 1 → `z` mit Streuung ≈ √61 ≈ 7,8 → Sigmoid fast überall 0 oder 1 (gesättigt), ReLU reicht Werte um ±10 weiter. Mit 0,1 → Streuung von `z` ≈ 0,8, Aktivierungen bleiben im sichtbaren Bereich. `rng.normal(scale=0.1, ...)` ist dasselbe wie `rng.normal(...) * 0.1`.
**Noch nicht ganz verstanden (Alp)** → Claude führt es Mi 04.11. am Notebook vor: dieselbe Rechnung mit Faktor 1 und 0,1, Vergleich `A2.min()` / `A2.max()`.

## 13 · Merksätze für Anki (Portion 26)

- Neuron = (w, b, g); Schicht = W (Eingänge, Neuronen) + b (Neuronen,); Netz = Schichten hintereinander.
- `(n, e) @ (e, k) → (n, k)`; Kredite kommen in W nie vor.
- Nebeneinander → neue Spalte `(4, 2)`, nicht `(8,)`.
- Ausgabe-Neuron einer Schicht = Eingabe-Merkmal der nächsten.
- Mitte darf breiter sein als die Eingabe; nur die letzte Schicht ist festgelegt.
- ReLU = `np.maximum(0, z)`; Sigmoid am Ausgang für ja/nein.
- `coef_` (1, 61) = `W.T` eines Ein-Neuron-Netzes.
- Zufallsgewichte → Münzwurf; Accuracy kommt aus dem Training.
